# Week 2 Updated Preprocessing Pipeline

This version reproduces the requested Week 2 feature structure and does **not**
include sentiment features.

Expected input:

```text
data/week1_initial_raw_dataset.csv
```

Generated output:

```text
output/week2_feature_dataset_pipeline.csv
```


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", None)


In [ ]:
def load_jpm_dividend_history(
    start_date: pd.Timestamp,
    end_date: pd.Timestamp,
) -> pd.DataFrame:
    """Download JPM dividend events and return a daily step-series."""

    ticker = yf.Ticker("JPM")
    dividend_events = ticker.dividends

    if dividend_events.empty:
        raise ValueError("No JPM dividend history was returned by yfinance.")

    dividend_events = dividend_events.copy()
    dividend_events.index = pd.to_datetime(dividend_events.index)

    # Remove timezone information for safe merging.
    if dividend_events.index.tz is not None:
        dividend_events.index = dividend_events.index.tz_localize(None)

    dividend_events = dividend_events.loc[
        (dividend_events.index <= end_date)
    ]

    # Keep the latest dividend known on each ex-dividend date.
    dividend_df = (
        dividend_events
        .rename("Dividend")
        .reset_index()
    )
    dividend_df.columns = ["Date", "Dividend"]
    dividend_df["Date"] = pd.to_datetime(dividend_df["Date"]).dt.normalize()

    return dividend_df


In [ ]:
def build_week2_pipeline(
    input_file: str,
    output_file: str,
) -> pd.DataFrame:
    """
    Create the requested Week 2 feature dataset.

    Features:
    - Daily_Return
    - Log_Return
    - VIX_Change
    - Rate_Change
    - Volume_Change
    - Rolling_Vol_5D
    - Rolling_Vol_20D
    - Rolling_Vol_60D
    - Dividend
    - Dividend_Growth
    - VIX_Return
    - VIX_JPM_Correlation_20D
    - Rate_Momentum_1D
    - Rate_Momentum_5D
    - Rate_Momentum_20D
    - Rate_Pct_Change_5D
    - Rate_Pct_Change_20D
    """

    input_path = Path(input_file)
    output_path = Path(output_file)

    if not input_path.exists():
        raise FileNotFoundError(
            f"Input file was not found: {input_path.resolve()}"
        )

    df = pd.read_csv(input_path)
    input_rows = len(df)

    # Preserve the requested display name "Adj Close".
    df.columns = [str(column).strip() for column in df.columns]

    if "Adj_Close" in df.columns and "Adj Close" not in df.columns:
        df = df.rename(columns={"Adj_Close": "Adj Close"})

    required_columns = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
    ]

    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing_columns)
        )

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = (
        df.dropna(subset=["Date"])
          .sort_values("Date")
          .drop_duplicates(subset=["Date"], keep="last")
          .reset_index(drop=True)
    )

    numeric_columns = [
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
    ]

    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    # Forward-fill market variables where appropriate.
    fill_columns = [
        column for column in [
            "Open",
            "High",
            "Low",
            "Close",
            "Adj Close",
            "Volume",
            "VIX",
            "Treasury_10Y",
        ]
        if column in df.columns
    ]
    df[fill_columns] = df[fill_columns].ffill()

    # Return and change features.
    df["Daily_Return"] = df["Close"].pct_change()
    df["Log_Return"] = np.log(df["Close"] / df["Close"].shift(1))

    df["VIX_Change"] = df["VIX"].pct_change()
    df["Rate_Change"] = df["Treasury_10Y"].diff()
    df["Volume_Change"] = df["Volume"].pct_change()

    # Annualized rolling historical volatility.
    for window in [5, 20, 60]:
        df[f"Rolling_Vol_{window}D"] = (
            df["Log_Return"]
            .rolling(window=window)
            .std()
            * np.sqrt(252)
        )

    # JPM dividend feature.
    dividend_events = load_jpm_dividend_history(
        start_date=df["Date"].min(),
        end_date=df["Date"].max(),
    )

    df = pd.merge_asof(
        df.sort_values("Date"),
        dividend_events.sort_values("Date"),
        on="Date",
        direction="backward",
    )

    # The dividend is treated as the latest announced quarterly amount.
    df["Dividend"] = df["Dividend"].ffill().fillna(0)

    # Growth changes only when the quarterly dividend amount changes.
    dividend_change = df["Dividend"].ne(df["Dividend"].shift(1))
    df["Dividend_Growth"] = 0.0

    previous_dividend = df["Dividend"].shift(1)
    valid_growth = dividend_change & previous_dividend.gt(0)

    df.loc[valid_growth, "Dividend_Growth"] = (
        df.loc[valid_growth, "Dividend"]
        / previous_dividend.loc[valid_growth]
        - 1
    )

    # VIX return and rolling JPM/VIX relationship.
    df["VIX_Return"] = df["VIX"].pct_change()

    df["VIX_JPM_Correlation_20D"] = (
        df["Daily_Return"]
        .rolling(window=20)
        .corr(df["VIX_Return"])
    )

    # Interest-rate momentum features.
    df["Rate_Momentum_1D"] = df["Treasury_10Y"].diff(1)
    df["Rate_Momentum_5D"] = df["Treasury_10Y"].diff(5)
    df["Rate_Momentum_20D"] = df["Treasury_10Y"].diff(20)

    df["Rate_Pct_Change_5D"] = df["Treasury_10Y"].pct_change(5)
    df["Rate_Pct_Change_20D"] = df["Treasury_10Y"].pct_change(20)

    df = df.replace([np.inf, -np.inf], np.nan)

    # Match the requested final column order.
    final_columns = [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "Adj Close",
        "Volume",
        "VIX",
        "Treasury_10Y",
        "Daily_Return",
        "Log_Return",
        "VIX_Change",
        "Rate_Change",
        "Volume_Change",
        "Rolling_Vol_5D",
        "Rolling_Vol_20D",
        "Rolling_Vol_60D",
        "Dividend",
        "Dividend_Growth",
        "VIX_Return",
        "VIX_JPM_Correlation_20D",
        "Rate_Momentum_1D",
        "Rate_Momentum_5D",
        "Rate_Momentum_20D",
        "Rate_Pct_Change_5D",
        "Rate_Pct_Change_20D",
    ]

    # Adj Close may not exist if the data source omitted it.
    final_columns = [
        column for column in final_columns
        if column in df.columns
    ]

    # The 60-day rolling feature determines the valid starting date.
    required_feature_columns = [
        "Daily_Return",
        "Log_Return",
        "Rolling_Vol_60D",
        "VIX_JPM_Correlation_20D",
        "Rate_Momentum_20D",
        "Rate_Pct_Change_20D",
    ]

    df = (
        df.dropna(subset=required_feature_columns)
          .loc[:, final_columns]
          .reset_index(drop=True)
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    print("Updated Week 2 pipeline completed successfully.")
    print(f"Input rows: {input_rows}")
    print(f"Output rows: {len(df)}")
    print(f"Output columns: {len(df.columns)}")
    print(f"Saved to: {output_path.resolve()}")

    return df


## Run the updated pipeline


In [ ]:
INPUT_FILE = "data/week1_initial_raw_dataset.csv"
OUTPUT_FILE = "output/week2_feature_dataset_pipeline.csv"

week2_data_pipeline = build_week2_pipeline(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
)


In [ ]:
display(week2_data_pipeline.head())
display(week2_data_pipeline.tail())


In [ ]:
quality_report = pd.DataFrame({
    "Column": week2_data_pipeline.columns,
    "Data_Type": week2_data_pipeline.dtypes.astype(str).values,
    "Missing_Values": week2_data_pipeline.isna().sum().values,
    "Unique_Values": week2_data_pipeline.nunique().values,
})

display(quality_report)


In [ ]:
expected_columns = [
    "Date",
    "Open",
    "High",
    "Low",
    "Close",
    "Adj Close",
    "Volume",
    "VIX",
    "Treasury_10Y",
    "Daily_Return",
    "Log_Return",
    "VIX_Change",
    "Rate_Change",
    "Volume_Change",
    "Rolling_Vol_5D",
    "Rolling_Vol_20D",
    "Rolling_Vol_60D",
    "Dividend",
    "Dividend_Growth",
    "VIX_Return",
    "VIX_JPM_Correlation_20D",
    "Rate_Momentum_1D",
    "Rate_Momentum_5D",
    "Rate_Momentum_20D",
    "Rate_Pct_Change_5D",
    "Rate_Pct_Change_20D",
]

missing_expected = [
    column for column in expected_columns
    if column not in week2_data_pipeline.columns
]

assert not missing_expected, (
    f"Missing expected output columns: {missing_expected}"
)

assert Path(OUTPUT_FILE).exists(), (
    f"Output CSV was not generated: {OUTPUT_FILE}"
)

print("Output structure verified successfully.")
